<a href="https://colab.research.google.com/github/jygheo/Audio-Sheet-Music-Retrieval/blob/main/text_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
from google.colab import runtime
drive.mount('/content/drive')

In [ ]:
!pip install mauve-text nltk

In [ ]:
import json
import torch
from transformers import AutoModel, AutoTokenizer
import nltk
from collections import Counter
import mauve
import numpy as np
import os

nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
def calculate_diversity(texts):
    """
    Calculates rep-2, rep-3, rep-4, and the final Diversity score.
    Diversity = (1 - rep-2) * (1 - rep-3) * (1 - rep-4)
    """
    rep_n = {2: 0.0, 3: 0.0, 4: 0.0}

    for n in [2, 3, 4]:
        total_ngrams = 0
        unique_ngrams = 0

        for text in texts:
            tokens = nltk.word_tokenize(text.lower())

            if len(tokens) < n:
                continue

            ngrams = [tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)]

            total_ngrams += len(ngrams)
            unique_ngrams += len(set(ngrams))

        if total_ngrams > 0:
            rep_n[n] = 1.0 - (unique_ngrams / total_ngrams)

    diversity = (1.0 - rep_n[2]) * (1.0 - rep_n[3]) * (1.0 - rep_n[4])
    return diversity, rep_n



def calculate_coherence_simcse(prompts, generations_text, tokenizer, model, batch_size=64):
    """
    Calculates the average cosine similarity between the prompt and the continuation
    using BERT-base
    """
    model.eval()
    all_similarities = []

    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i : i + batch_size]
        batch_gens = generations_text[i : i + batch_size]
        inputs_prompts = tokenizer(batch_prompts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(model.device)
        inputs_gens = tokenizer(batch_gens, padding=True, truncation=True, max_length=128, return_tensors="pt").to(model.device)

        with torch.no_grad():
            emb_prompts = model(**inputs_prompts).pooler_output
            emb_gens = model(**inputs_gens).pooler_output
            emb_prompts = emb_prompts / emb_prompts.norm(dim=1, keepdim=True)
            emb_gens = emb_gens / emb_gens.norm(dim=1, keepdim=True)
            batch_sims = (emb_prompts * emb_gens).sum(dim=1)
            all_similarities.extend(batch_sims.cpu().tolist())

    return sum(all_similarities) / len(all_similarities)


def calculate_mauve(references_text, generations_text, tokenizer):
    target_length = 256
    reference_token_ids = tokenizer(
        references_text,
        truncation=True,
        max_length=target_length
    )['input_ids']
    generation_token_ids = tokenizer(
        generations_text,
        truncation=True,
        max_length=target_length
    )['input_ids']
    valid_pairs = [
        (reference_ids, generation_ids)
        for (reference_ids, generation_ids) in zip(reference_token_ids, generation_token_ids)
        if len(reference_ids) == target_length and len(generation_ids) == target_length
    ]
    if not valid_pairs:
        return 0.0
    reference_token_ids, generation_token_ids = zip(*valid_pairs)
    print(
        f"  -> MAUVE Filter: Kept "
        f"{len(reference_token_ids)}/{len(generations_text)} pairs"
    )
    decoded_references = tokenizer.batch_decode(reference_token_ids)
    decoded_generations = tokenizer.batch_decode(generation_token_ids)
    mauve_output = mauve.compute_mauve(
        p_text=decoded_references,
        q_text=decoded_generations,
        device_id=0,
        max_text_length=256,
        verbose=False,
        featurize_model_name='gpt2',
    )
    return mauve_output.mauve

def evaluate_dataset(jsonl_path, tokenizer):
    print(f"Loading data from: {jsonl_path}")

    prompts = []
    references_tokens = []
    method_generations = {}

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            prompts.append(record["prompt"])
            references_tokens.append(record["human_reference"])

            for method, token_list in record["generations"].items():
                if method not in method_generations:
                    method_generations[method] = []
                method_generations[method].append(token_list)

    total_samples = len(prompts)
    print(f"Loaded {total_samples} samples. Found methods: {list(method_generations.keys())}\n")

    simcse_tokenizer = AutoTokenizer.from_pretrained("princeton-nlp/sup-simcse-bert-base-uncased")
    simcse_model = AutoModel.from_pretrained("princeton-nlp/sup-simcse-bert-base-uncased").cuda()

    results = {}
    references_text = tokenizer.batch_decode(references_tokens, skip_special_tokens=True)
    for method, generations_tokens in method_generations.items():
        generations_text = tokenizer.batch_decode(generations_tokens, skip_special_tokens=True)

        # Diversity
        div_score, rep_n = calculate_diversity(generations_text)

        # Coherence
        coh_score = calculate_coherence_simcse(prompts, generations_text, simcse_tokenizer, simcse_model)

        # MAUVE
        mauve_score = calculate_mauve(references_text, generations_text, tokenizer)

        results[method] = {
            "Diversity": round(div_score, 3),
            "Coherence": round(coh_score, 3),
            "MAUVE": round(mauve_score, 3)
        }

    print(f"=== FINAL RESULTS for {jsonl_path} ===")
    print(f"{'Method':<15} | {'Diversity':<10} | {'MAUVE':<10} | {'Coherence':<10}")
    print("-" * 55)
    for method, metrics in results.items():
        print(f"{method:<15} | {metrics['Diversity']:<10} | {metrics['MAUVE']:<10} | {metrics['Coherence']:<10}")



In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# replace with path to .jsonl to evaluate
file_path = "/content/drive/MyDrive/"
evaluate_dataset(file_path, tokenizer)


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-125m")
tokenizer.pad_token = tokenizer.eos_token

# replace with path to .jsonl to evaluate
file_path = "/content/drive/MyDrive/"
evaluate_dataset(file_path, tokenizer)

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-7B")
tokenizer.pad_token = tokenizer.eos_token

# replace with path to .jsonl to evaluate
file_path = "/content/drive/MyDrive/"
evaluate_dataset(file_path, tokenizer)

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# replace with path to .jsonl to evaluate
file_path = "/content/drive/MyDrive/"
evaluate_dataset(file_path, tokenizer)